# Notebook 07 — Coherent GW reconstruction test (N=3, λ̄=0.84)

Uses the same 3-bubble configuration as nb05 to validate the coherent reconstruction
approach against the incoherent result and the 3+1D SH simulation.

**Strategy**: re-run BubbleMaster for each of the 3 pairs at their exact gamma with
`--save-amplitude`, verify self-consistency, then compare incoherent vs coherent.

In [ ]:
import subprocess
import types
import numpy as np
import h5py
import matplotlib.pyplot as plt
from pathlib import Path
from numpy.polynomial.legendre import leggauss
from scipy.interpolate import RegularGridInterpolator

import ptbuilder as pt
from ptbuilder.ic import write_2d_setup, BubbleMasterParams
from ptbuilder.physics import InstantonProfile
from ptbuilder.analysis import (load_weights, bubblemaster_normalization,
                                 load_amplitude_scan, check_amplitude_consistency,
                                 apply_keffsq)
from ptbuilder.scan import load_scan

REPO_ROOT   = Path(pt.__file__).parent.parent
DATA_DIR    = REPO_ROOT / 'data'
SH_RUNS_DIR = REPO_ROOT / 'sledgehamr_runs'
BM_BIN      = REPO_ROOT / 'cpp/bubblemaster/bubblemaster'
AMP_DIR     = DATA_DIR / 'amp_N3_084'

lb       = 0.84
L        = 200.
zero_pad = 2.

# Bubble positions (from nb05)
bubbles = np.array([[72.25, 89.92, 100.],
                    [127.8, 89.92, 100.],
                    [108.7, 110.1, 100.]])

PAIRS = [(0,1), (0,2), (1,2)]

In [12]:
# Build a lightweight model object from the stored instanton.h5.
# Avoids needing the compiled C++ _potential extension locally.
with h5py.File(DATA_DIR / f'phi4_lambda_bar{lb}' / 'instanton.h5') as _f:
    _profile = InstantonProfile(
        R=_f['R'][:], Phi=_f['Phi'][:],
        rin_0 =float(_f.attrs['rin_0']),
        rmid_0=float(_f.attrs['rmid_0']),
        rout_0=float(_f.attrs['rout_0']),
        interp=None,
    )
    _pa = dict(_f['potential'].attrs)

def _pot_to_hdf5(pot, grp):
    grp.attrs.create('type', pot.type_name, dtype=h5py.string_dtype(encoding='ascii'))
    grp.attrs['phi_false'] = float(pot.phi_false)
    grp.attrs['phi_true']  = float(pot.phi_true)
    for k, v in pot.params.items():
        grp.attrs[k] = float(v)

_potential = types.SimpleNamespace(
    type_name = _pa['type'],
    phi_false = float(_pa['phi_false']),
    phi_true  = float(_pa['phi_true']),
    params    = {'lambda_bar': float(_pa['lambda_bar'])},
)
_potential.to_hdf5 = lambda grp: _pot_to_hdf5(_potential, grp)
model = types.SimpleNamespace(instanton=_profile, potential=_potential)

# Time grid — must match the existing scan_cache
N_T        = 32
T_MAX      = 70.
scan_times = np.linspace(1., T_MAX, N_T)


In [13]:
# Pair weights
N_WEIGHTS     = 500
weights_times = np.linspace(1., T_MAX, N_WEIGHTS)

pair_weights = {}
pair_gammas  = {}
pair_d       = {}

for (i, j) in PAIRS:
    wout = DATA_DIR / f'weights_out_N3_084_pair_{i}_{j}.h5'
    w, _, _, gam = load_weights(wout, N_WEIGHTS)
    dp = bubbles[i] - bubbles[j]
    dp -= L * np.round(dp / L)
    d  = float(np.linalg.norm(dp))
    pair_weights[(i,j)] = w[0]
    pair_gammas [(i,j)] = float(gam[0])
    pair_d      [(i,j)] = d
    print(f'pair ({i},{j}): gamma={float(gam[0]):.4f}  d={d:.4f}')

pair (0,1): gamma=3.9739  d=55.5500
pair (0,2): gamma=2.9622  d=41.6634
pair (1,2): gamma=1.9397  d=27.7857


## 1. Generate setup files and SLURM scripts

In [ ]:
from ptbuilder.config import Config
config      = Config()
bm_params   = BubbleMasterParams()
scripts_dir = REPO_ROOT / 'scripts'
logs_dir    = REPO_ROOT / 'logs'
WALLTIME    = '04:00:00'
QOS         = 'premium'

AMP_DIR.mkdir(parents=True, exist_ok=True)

for (i, j) in PAIRS:
    setup_path = AMP_DIR / f'setup_{i}_{j}.h5'
    out_dir    = AMP_DIR / f'pair_{i}_{j}'
    tag        = f'amplitude_N3_084_pair_{i}_{j}'

    if not setup_path.exists():
        write_2d_setup(model, pair_gammas[(i,j)], scan_times, setup_path, bm_params)
        print(f'pair ({i},{j}): wrote {setup_path.name}')

    slurm_path = scripts_dir / f'{tag}.sh'
    slurm_path.write_text(
        '#!/bin/bash\n'
        f'#SBATCH --job-name={tag}\n'
        f'#SBATCH --output={logs_dir}/{tag}_%j.out\n'
        '#SBATCH --constraint=cpu\n'
        f'#SBATCH --qos={QOS}\n'
        '#SBATCH --nodes=1\n'
        '#SBATCH --tasks-per-node=1\n'
        '#SBATCH --cpus-per-task=128\n'
        f'#SBATCH --time={WALLTIME}\n\n'
        'export SLURM_CPU_BIND=cores\n'
        'export OMP_PLACES=threads\n'
        'export OMP_PROC_BIND=spread\n'
        'export OMP_NUM_THREADS=128\n'
        f'mkdir -p {out_dir}\n'
        f'/usr/bin/time --verbose srun {config.bubblemaster_bin} '
        f'{setup_path} {out_dir}/ --save-amplitude\n'
    )
    print(f'sbatch {slurm_path}')

In [ ]:
# Run after all SLURM jobs complete
amp_scans = {}
for (i, j) in PAIRS:
    out_dir = AMP_DIR / f'pair_{i}_{j}'
    if not out_dir.exists() or not any(out_dir.glob('result_*.h5')):
        raise FileNotFoundError(
            f"No amplitude data for pair ({i},{j}) — submit the SLURM job first."
        )
    amp_scans[(i,j)] = load_amplitude_scan(out_dir)
    print(f'pair ({i},{j}): amp_re shape = {amp_scans[(i,j)]["amp_re"].shape}')

## 2. Self-consistency check

Verify that squaring the stored amplitude and direction-integrating recovers the stored power spectrum.

In [ ]:
for (i, j) in PAIRS:
    check_amplitude_consistency(amp_scans[(i,j)])
    print(f'pair ({i},{j}): OK')

## 3. Effective amplitude per pair

Time-weight the incremental complex amplitudes to get `A_eff(w, cos_theta)` for each pair.

In [ ]:
k_bm = amp_scans[PAIRS[0]]['k']   # (n_k,) cos-theta grid, linspace(0,1,n_k)
w_bm = amp_scans[PAIRS[0]]['w']   # (n_w,) frequency grid
n_k  = len(k_bm)
n_w  = len(w_bm)

def A_eff_pair(amp_scan, weights_t):
    wp  = np.interp(scan_times, weights_times, weights_t)   # (n_t,)
    dar = np.diff(amp_scan['amp_re'], axis=0, prepend=0.)   # (n_t, n_w, n_k)
    dai = np.diff(amp_scan['amp_im'], axis=0, prepend=0.)
    return (wp[:, None, None] * dar).sum(axis=0), (wp[:, None, None] * dai).sum(axis=0)

A_effs = {}
for (i, j) in PAIRS:
    A_effs[(i,j)] = A_eff_pair(amp_scans[(i,j)], pair_weights[(i,j)])
    print(f'pair ({i},{j}): A_eff max = {np.abs(A_effs[(i,j)][0]).max():.3e}')

## 4. Incoherent reconstruction from amplitude

Sum `|A_eff|²` over pairs, then direction-integrate. Should match the nb05 result.

In [ ]:
dk = k_bm[1] - k_bm[0]
fk = np.full(n_k, 2.); fk[0] = fk[-1] = 1.   # trapezoid × symmetry factor for k ∈ [0,1]

incoh_amp = np.zeros(n_w)
for (i, j) in PAIRS:
    Ar, Ai = A_effs[(i,j)]
    incoh_amp += ((Ar**2 + Ai**2) * fk[None,:]).sum(axis=-1) * dk * w_bm**3 * 2*np.pi

kconv, bm_norm = bubblemaster_normalization(lb)
incoh_amp_normed = incoh_amp * bm_norm
print(f'Incoherent (amplitude): peak = {incoh_amp_normed.max():.3e}')

## 5. Coherent reconstruction

Sum amplitudes with position phase factors **before** squaring, then integrate over the full sphere.

$$P(\omega) = \int d\Omega\, \left|\sum_p A^{(p)}_{\rm eff}(\omega,\hat{k}\cdot\hat{n}_p)\, e^{i\omega\hat{k}\cdot x_p^{\rm cm}}\right|^2 \omega^3 \cdot 2\pi$$

In [ ]:
# Pair axes and center-of-mass positions (minimum-image PBC)
pair_axes    = {}
pair_centers = {}
for (i, j) in PAIRS:
    dp = bubbles[i] - bubbles[j]
    dp -= L * np.round(dp / L)
    pair_axes[(i,j)]    = dp / np.linalg.norm(dp)
    pair_centers[(i,j)] = (0.5 * (bubbles[i] + bubbles[j])) % L

# Gauss-Legendre × uniform-phi quadrature on the sphere
N_THETA, N_PHI = 64, 128
cos_nodes, gl_w = leggauss(N_THETA)
phi_vals = np.linspace(0., 2.*np.pi, N_PHI, endpoint=False)
d_phi    = 2.*np.pi / N_PHI

CT, PH = np.meshgrid(cos_nodes, phi_vals, indexing='ij')   # (N_THETA, N_PHI)
ST     = np.sqrt(np.maximum(1. - CT**2, 0.))
hat_k  = np.stack([ST*np.cos(PH), ST*np.sin(PH), CT], axis=-1).reshape(-1, 3)  # (N_sph, 3)
d_omega = np.outer(gl_w, np.full(N_PHI, d_phi)).ravel()   # (N_sph,)

N_sph = len(hat_k)
print(f'Sphere quadrature: {N_sph} points')

In [ ]:
def interp_amp_all_dirs(A, cos_th_all):
    # A: (n_w, n_k), cos_th_all: (N_sph,) — vectorized linear interp along k axis
    # returns (n_w, N_sph)
    idx = np.clip(np.searchsorted(k_bm, cos_th_all), 1, n_k - 1)   # (N_sph,)
    t   = (cos_th_all - k_bm[idx-1]) / (k_bm[idx] - k_bm[idx-1])  # (N_sph,)
    return A[:, idx-1] * (1 - t)[None,:] + A[:, idx] * t[None,:]    # (n_w, N_sph)

A_tot_re = np.zeros((n_w, N_sph))
A_tot_im = np.zeros((n_w, N_sph))

for (i, j) in PAIRS:
    Ar, Ai   = A_effs[(i,j)]
    # cos(theta) between each sphere direction and this pair's collision axis
    cos_th   = np.clip(np.abs(hat_k @ pair_axes[(i,j)]), 0., 1.)   # (N_sph,)
    Ap_re    = interp_amp_all_dirs(Ar, cos_th)   # (n_w, N_sph)
    Ap_im    = interp_amp_all_dirs(Ai, cos_th)
    # phase: w * hat_k · x_cm    shape (n_w, N_sph)
    ph       = np.outer(w_bm, hat_k @ pair_centers[(i,j)])
    cp, sp   = np.cos(ph), np.sin(ph)
    A_tot_re += Ap_re * cp - Ap_im * sp
    A_tot_im += Ap_re * sp + Ap_im * cp

# direction integral: sum over N_sph with quadrature weights
coh = ((A_tot_re**2 + A_tot_im**2)
       * (w_bm**3 * 2.*np.pi)[:, None]
       * d_omega[None, :]).sum(axis=-1)

coh_normed = coh * bm_norm
print(f'Coherent reconstruction: peak = {coh_normed.max():.3e}')

## 6. Incoherent from existing scan (nb05 reference)

In [ ]:
scan_results = load_scan(DATA_DIR / f'phi4_lambda_bar{lb}' / 'scan_cache.h5')
gammas_grid  = np.array(sorted(scan_results.keys()))
first_sr     = next(iter(scan_results.values()))
w_scan       = first_sr.w
n_w_scan     = len(w_scan)
N_T_scan     = len(first_sr.spectrum)

spec_grid = np.zeros((len(gammas_grid), N_T_scan, n_w_scan))
for ig, gamma in enumerate(gammas_grid):
    spec_grid[ig] = scan_results[gamma].spectrum

floor = spec_grid.max() * 1e-12
_raw  = RegularGridInterpolator(
    (gammas_grid, scan_times[:N_T_scan]),
    np.log(np.maximum(spec_grid, floor)),
    bounds_error=False, fill_value=np.log(floor),
)

def bm_reconstruct_scan(gamma, weights_t):
    g   = float(np.clip(gamma, gammas_grid.min(), gammas_grid.max()))
    tc  = scan_times[:N_T_scan]
    wp  = np.interp(tc, weights_times, weights_t)
    pts = np.column_stack([np.full(len(tc), g), tc])
    sp  = np.exp(_raw(pts))
    dsp = np.diff(sp, axis=0)
    return (wp[:len(tc)-1, None] * dsp).sum(axis=0)

incoh_scan = np.zeros(n_w_scan)
for (i, j) in PAIRS:
    incoh_scan += bm_reconstruct_scan(pair_gammas[(i,j)], pair_weights[(i,j)])
incoh_scan_normed = incoh_scan * bm_norm
print(f'Incoherent (scan interp): peak = {incoh_scan_normed.max():.3e}')

## 7. Comparison plot

In [ ]:
import glob as _glob

pair_sh_paths = {
    (0,1): SH_RUNS_DIR / 'bubble_N3_234_0_1_',
    (0,2): SH_RUNS_DIR / 'bubble_N3_234_0_2_',
    (1,2): SH_RUNS_DIR / 'bubble_normalization_gamma2',
}
sh_3b_path = SH_RUNS_DIR / 'bubble_N3_234_3b_084'

def _list_gw_files(sh_path):
    entries = []
    for fp in _glob.glob(str(sh_path / 'gw_spectra' / '*' / 'spectra.hdf5')):
        dname = Path(fp).parent.name
        if dname.isdigit():
            entries.append((int(dname), fp))
    entries.sort()
    return entries

def _read_gw_snap(fp):
    with h5py.File(fp, 'r') as f:
        k_int = np.asarray(f['k' if 'k' in f else 'k_sq'][:])
        spec  = np.asarray(f['Spectrum' if 'Spectrum' in f else 'spectrum'][:])
        hdr   = np.asarray(f['Header'][:]).reshape(-1)
        t     = float(hdr[0])
    return apply_keffsq(k_int), spec, t

def load_sh_snaps(sh_path):
    files = _list_gw_files(sh_path)
    if not files:
        raise FileNotFoundError(f'No GW snapshots in {sh_path}')
    snaps = [_read_gw_snap(fp) for _, fp in files]
    snaps.sort(key=lambda x: x[2])
    return snaps

def normalize_sh_local(k_sq, spectrum):
    if k_sq[0] == 0:
        k_sq = k_sq[1:]; spectrum = spectrum[1:]
    r    = np.sqrt(k_sq)
    k    = r * (2 * np.pi / (L * zero_pad))
    norm = 4.0 * (2 * np.pi * r) * (L * zero_pad)**3
    return k, spectrum * norm

sh_pair_last = {}
for ij, path in pair_sh_paths.items():
    try:
        snaps = load_sh_snaps(path)
        sh_pair_last[ij] = snaps[-1]
        print(f'pair {ij}: {len(snaps)} snaps, last t={snaps[-1][2]:.1f}')
    except Exception as e:
        print(f'pair {ij}: {e}')

try:
    sh_3b_snaps = load_sh_snaps(sh_3b_path)
    sh_3b_last  = sh_3b_snaps[-1]
    print(f'3-bubble: {len(sh_3b_snaps)} snaps, last t={sh_3b_last[2]:.1f}')
except Exception as e:
    print(f'3-bubble: {e}')
    sh_3b_last = None

In [ ]:
k_bm_phys   = w_bm  * kconv
k_scan_phys = w_scan * kconv
colors_pair = ['#0072B2', '#D55E00', '#009E73']

fig, ax = plt.subplots(figsize=(9, 5))

# Individual 2-bubble SH runs (final snapshot)
for color, ij in zip(colors_pair, PAIRS):
    if ij in sh_pair_last:
        k_sq, spec, t = sh_pair_last[ij]
        k_ph, y_ph    = normalize_sh_local(k_sq, spec)
        ax.plot(k_ph, y_ph, color=color, lw=1.5, ls='-',
                label=f'3D pair {ij}  t={t:.0f}')

# 3-bubble SH run (final snapshot)
if sh_3b_last is not None:
    k_sq, spec, t = sh_3b_last
    k_ph, y_ph    = normalize_sh_local(k_sq, spec)
    ax.plot(k_ph, y_ph, color='black', lw=2, ls='-',
            label=f'3D 3-bubble  t={t:.0f}')

# BM reconstructions
ax.plot(k_scan_phys, incoh_scan_normed, color='C3', lw=2, ls='--',
        label='Incoherent (scan interp)')
ax.plot(k_bm_phys,   incoh_amp_normed,  color='C4', lw=2, ls='-.',
        label='Incoherent (amplitude)')
ax.plot(k_bm_phys,   coh_normed,        color='C5', lw=2, ls=':',
        label='Coherent')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r'$k$')
ax.set_ylabel(r'$dE_{\rm GW}/d\ln k$')
ax.legend(frameon=False)
ax.set_title(rf'N=3 coherent reconstruction — $\bar{{\lambda}}={lb}$')
plt.tight_layout()
plt.show()